# Multilayer Perceptron

Dataset: MNIST  
Each piece of data is a 28*28(784 features) grayscale image of a handwritten digit. Training-set includes 60000 images; Test-set includes 10000 images

Number of input layer neuron: 784 (size: 28*28)    
Number of output layer neuron: 10 (class: 10)

### Data processing

In [66]:
import pandas as pd
import torch


In [67]:
# complete the code here: 
# Read .csv files to load data
train_path = './mnist_train.csv'
test_path = './mnist_test.csv'
train_data = pd.read_csv(train_path,header=None)
test_data = pd.read_csv(test_path,header=None)

In [68]:
# complete the code here: 
# 1. print the shape of the train and test data 
# 2. explain the meaning of shapes in the comments
print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
#(60000,785) means the train data has 60000 samples. Each sample has 785 dimensions
#(10000,785) means the train data has 10000 samples. Each sample has 785 dimensions

Train data shape: (60000, 785)
Test data shape: (10000, 785)


In [69]:
y_train = train_data.iloc[:, 0].values  # get labels
x_train = train_data.iloc[:, 1:].values # get features

y_test = test_data.iloc[:, 0].values    # get labels
x_test = test_data.iloc[:, 1:].values   # get features

In [70]:
# convert data to tensor
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [71]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

torch.Size([60000, 784])
torch.Size([60000])
torch.Size([10000, 784])
torch.Size([10000])


### Building Neural Networks

In [72]:
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [74]:
# explain the details of the neural network designed here by adding comments

class MNIST(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 1000)  #define fc1, transform 28*28 input features to 1000 neurons
        self.fc2 = nn.Linear(1000, 500)    #define fc2, transform 1000 input features to 500 neurons
        self.fc3 = nn.Linear(500, 10)      #define fc3, transform 500 input features to 10 neurons
    
    def forward(self, x):
        x = F.relu(self.fc1(x))    #apply fc2 with ReLU
        x = F.relu(self.fc2(x))    #apply fc2 with ReLU
        x = self.fc3(x)            #get logits from fc3
        return x



In [75]:
mnist = MNIST().to(device) # instantiation
print(mnist)

MNIST(
  (fc1): Linear(in_features=784, out_features=1000, bias=True)
  (fc2): Linear(in_features=1000, out_features=500, bias=True)
  (fc3): Linear(in_features=500, out_features=10, bias=True)
)


In [76]:
# this code blcok is the training process
# define learning rate
lr = 0.001

losses = []

for epoch in range(100):
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    y_pred = mnist(x_train)
    
    # calculate the loss
    loss = F.cross_entropy(y_pred, y_train)

    # backpropagation
    loss.backward()

    # Updating the gradient
    with torch.no_grad():
        for p in mnist.parameters():
            p -= lr * p.grad
        mnist.zero_grad() # Avoid gradient accumulation
        
        print(f'Epoch {epoch}, Loss: {loss.item()}')
        losses.append(loss.item())
        

Epoch 0, Loss: 12.912888526916504
Epoch 1, Loss: 15.544703483581543
Epoch 2, Loss: 15.342495918273926
Epoch 3, Loss: 11.702773094177246
Epoch 4, Loss: 7.370079040527344
Epoch 5, Loss: 3.8304085731506348
Epoch 6, Loss: 2.326746702194214
Epoch 7, Loss: 1.7395131587982178
Epoch 8, Loss: 1.5544075965881348
Epoch 9, Loss: 1.3967891931533813
Epoch 10, Loss: 1.1979504823684692
Epoch 11, Loss: 1.1019439697265625
Epoch 12, Loss: 1.0303704738616943
Epoch 13, Loss: 0.9875863194465637
Epoch 14, Loss: 0.9475403428077698
Epoch 15, Loss: 0.914079487323761
Epoch 16, Loss: 0.8839377164840698
Epoch 17, Loss: 0.8545557856559753
Epoch 18, Loss: 0.8295456767082214
Epoch 19, Loss: 0.8056088089942932
Epoch 20, Loss: 0.7850815057754517
Epoch 21, Loss: 0.7661139369010925
Epoch 22, Loss: 0.7492663264274597
Epoch 23, Loss: 0.7337700128555298
Epoch 24, Loss: 0.7195620536804199
Epoch 25, Loss: 0.7063411474227905
Epoch 26, Loss: 0.6939889192581177
Epoch 27, Loss: 0.6823478937149048
Epoch 28, Loss: 0.671335995197296

In [77]:
# explain the purpose of this code block by adding comments
# this code block evaluates the performance of the trained model
with torch.no_grad():
    x_test = x_test.to(device)
    y_test = y_test.to(device)

    y_test_pred = mnist(x_test)  
    _, predicted = torch.max(y_test_pred, 1)  
    correct = (predicted == y_test).sum().item()  
    accuracy = correct / len(y_test)  
    print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 89.74%


In [85]:
import numpy as np
print("\nTest Cases:")
num_samples = 5 
for i in range(num_samples):
    input_data = x_test[i].squeeze().numpy() 
    predicted_label = predicted[i].item()
    formatted_input = input_data.round().astype(int).tolist()
    print(f"Sample {i+1}")
    print("Input number: ")
    print(np.array(formatted_input))
    print(f"Prediction result: {predicted_label}\n")


Test Cases:
Sample 1
Input number: 
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 